# 04d — Final test evaluation (Week 4 wrap-up)

**This notebook touches the test set.** It runs PhoBERT and BiLSTM v2
exactly once on TEST, joins the result with the Week-3 LR test metrics,
and writes the Week-4 final summary.

**Rule, not optional:** do NOT tune anything after seeing these numbers.

Parts:
1. **A** — PhoBERT on TEST (predict, metrics, confusion matrix, save predictions).
2. **B** — Dev vs Test generalisation gap for PhoBERT.
3. **C** — BiLSTM v2 on TEST; 3-way comparison (LR vs BiLSTM v2 vs PhoBERT).
4. **D** — Write `report/week4_final_summary.md` + `results/week4_test_metrics.json`.

**Outputs:**

* `results/phobert_test_predictions.csv`
* `results/bilstm_v2_test_predictions.csv`
* `results/week4_test_metrics.json`
* `results/figures/21_phobert_test_cm.png`
* `results/figures/22_three_way_test_comparison.png`
* `report/week4_final_summary.md`


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

import sys
for _k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path = [str(ROOT)] + [p for p in sys.path if p != str(ROOT)]

from configs.config import PATHS, COLUMNS, LABEL_MAP, PHOBERT_CONFIG, DEEP_LEARNING_CONFIG
from src.utils import set_seed, get_device

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = get_device()

DL_DIR        = ROOT / "models" / "dl"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR   = ROOT / "results"
FIG_DIR       = RESULTS_DIR / "figures"
REPORT_DIR    = ROOT / "report"
for d in (RESULTS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEXT, LABEL = COLUMNS["text"], COLUMNS["label"]

test_df = pd.read_csv(PROCESSED_DIR / "test_cleaned.csv")
print(f"Device : {device}")
print(f"Test set: {len(test_df):,} samples")
print(f"Test class distribution: {test_df[LABEL].value_counts().sort_index().to_dict()}")


## A. PhoBERT on TEST

Reload the saved best model (proves the artefact is portable), build the
PhoBERT-mode test dataset **without filtering**, predict with fp16
inference, then compute metrics + confusion matrix + save predictions.

In [ ]:
PHOBERT_BEST_DIR = DL_DIR / "phobert_best"
assert PHOBERT_BEST_DIR.exists(), f"Missing {PHOBERT_BEST_DIR} — run 04c first."

print(f"Loading model from {PHOBERT_BEST_DIR}")
phobert_model = AutoModelForSequenceClassification.from_pretrained(str(PHOBERT_BEST_DIR)).to(device)
phobert_tok   = AutoTokenizer.from_pretrained(str(PHOBERT_BEST_DIR), use_fast=False)
phobert_model.eval()

if device.type == "cuda":
    print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

PHOBERT_PARAMS = sum(p.numel() for p in phobert_model.parameters())
print(f"Total params: {PHOBERT_PARAMS:,}")


In [ ]:
from src.dataset_dl import ViHSDDataset, collate_fn_phobert

test_ds_phobert = ViHSDDataset(
    texts     = test_df["cleaned"].fillna("").tolist(),
    labels    = test_df[LABEL].tolist(),
    tokenizer = phobert_tok,
    max_len   = PHOBERT_CONFIG["max_len"],     # 128
    mode      = "phobert",
)
print(f"PhoBERT test_ds: {len(test_ds_phobert):,} samples  (unfiltered — fair eval)")


In [ ]:
# Predict on test with fp16 inference for speed.
TEST_BATCH = 64

# Windows + Jupyter: keep num_workers=0 to avoid spawn hangs.
test_loader = DataLoader(
    test_ds_phobert, batch_size=TEST_BATCH, shuffle=False,
    collate_fn=collate_fn_phobert, num_workers=0, pin_memory=True,
)

all_preds, all_labels, all_probs = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="phobert test"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attn      = batch["attention_mask"].to(device, non_blocking=True)
        labels    = batch["labels"]
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = phobert_model(input_ids=input_ids, attention_mask=attn)
        logits = out.logits.float()
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        preds  = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

phobert_test_time = time.perf_counter() - t0
y_test_true       = np.asarray(all_labels)
y_test_pred_phb   = np.asarray(all_preds)
y_test_proba_phb  = np.asarray(all_probs)

print(f"\nPhoBERT TEST inference: {phobert_test_time:.2f}s on {len(y_test_true):,} samples "
      f"({phobert_test_time/len(y_test_true)*1000:.2f} ms/sample)")

from src.evaluate import evaluate_model, plot_confusion_matrix

phobert_test_metrics = evaluate_model(
    y_true         = y_test_true,
    y_pred         = y_test_pred_phb,
    model_name     = "PhoBERT-base-v2 (fine-tuned) — TEST",
    train_time     = 0.0,
    inference_time = float(phobert_test_time),
)

print(f"\nPhoBERT TEST metrics:")
print(f"  accuracy : {phobert_test_metrics['accuracy']:.4f}")
print(f"  f1_macro : {phobert_test_metrics['f1_macro']:.4f}")
print(f"  f1 clean / off / hate : "
      f"{phobert_test_metrics['f1_clean']:.4f} / "
      f"{phobert_test_metrics['f1_offensive']:.4f} / "
      f"{phobert_test_metrics['f1_hate']:.4f}")
print()
print(phobert_test_metrics["classification_report"])

cm_path = FIG_DIR / "21_phobert_test_cm.png"
plot_confusion_matrix(
    y_true=y_test_true, y_pred=y_test_pred_phb,
    model_name="PhoBERT-base-v2 — TEST",
    save_path=cm_path, normalize=True,
)
plt.show()
print(f"✓ saved → {cm_path}")


In [ ]:
phobert_test_pred_df = pd.DataFrame({
    "text_original":    test_df[TEXT].values,
    "text_cleaned":     test_df["cleaned"].fillna("").values,
    "true_label":       y_test_true,
    "true_label_name":  [LABEL_MAP[int(l)] for l in y_test_true],
    "pred_label":       y_test_pred_phb,
    "pred_label_name":  [LABEL_MAP[int(l)] for l in y_test_pred_phb],
    "correct":          (y_test_true == y_test_pred_phb),
    "confidence":       y_test_proba_phb.max(axis=1).round(4),
    "prob_clean":       y_test_proba_phb[:, 0].round(4),
    "prob_offensive":   y_test_proba_phb[:, 1].round(4),
    "prob_hate":        y_test_proba_phb[:, 2].round(4),
})
phobert_test_csv = RESULTS_DIR / "phobert_test_predictions.csv"
phobert_test_pred_df.to_csv(phobert_test_csv, index=False)
print(f"✓ saved {len(phobert_test_pred_df):,} predictions → {phobert_test_csv}")
print(f"  correct: {int(phobert_test_pred_df['correct'].sum()):,} / {len(phobert_test_pred_df):,} "
      f"({phobert_test_pred_df['correct'].mean()*100:.2f}%)")


## B. PhoBERT dev vs test — generalisation gap

If dev metrics are far from test (>0.04 absolute f1_macro), we
overfit to dev. If they're close, generalisation is healthy.

In [ ]:
# Re-compute PhoBERT dev metrics from the saved CSV (single source of truth).
DEV_CSV = RESULTS_DIR / "phobert_dev_predictions.csv"
assert DEV_CSV.exists(), f"Missing {DEV_CSV} — run 04c first."

dev_pred_df = pd.read_csv(DEV_CSV)
# 04c used save_predictions() → schema: predicted_label, proba_*
y_dev_true = dev_pred_df["true_label"].to_numpy()
y_dev_pred = dev_pred_df["predicted_label"].to_numpy()

phobert_dev_metrics = evaluate_model(
    y_true=y_dev_true, y_pred=y_dev_pred,
    model_name="PhoBERT-base-v2 — DEV",
    train_time=0.0, inference_time=0.0,
)

# Build dev vs test table.
def _delta_rows(dev, test, keys):
    rows = []
    for k, label in keys:
        d, t = float(dev[k]), float(test[k])
        rows.append({
            "Metric":      label,
            "Dev":         d,
            "Test":        t,
            "Δ_abs":       t - d,
            "Δ_rel_pct":   ((t - d) / d * 100) if d else 0.0,
        })
    return rows

dev_vs_test_keys = [
    ("f1_macro",        "F1_macro"),
    ("f1_clean",        "F1_CLEAN"),
    ("f1_offensive",    "F1_OFFENSIVE"),
    ("f1_hate",         "F1_HATE"),
    ("accuracy",        "Accuracy"),
    ("precision_macro", "Precision_macro"),
    ("recall_macro",    "Recall_macro"),
]
dvt_rows = _delta_rows(phobert_dev_metrics, phobert_test_metrics, dev_vs_test_keys)
dvt_df = pd.DataFrame(dvt_rows)

print("PhoBERT — Dev vs Test:\n")
print(dvt_df.to_string(
    index=False,
    formatters={
        "Dev":       "{:.4f}".format,
        "Test":      "{:.4f}".format,
        "Δ_abs":     "{:+.4f}".format,
        "Δ_rel_pct": "{:+.2f}%".format,
    }))

# Verdict.
gap_macro = abs(dvt_rows[0]["Δ_abs"])
if gap_macro < 0.02:
    gen_verdict = "Excellent generalization"
elif gap_macro < 0.04:
    gen_verdict = "Good generalization, mild gap"
else:
    gen_verdict = "Overfit concern — investigate in Week-5 error analysis"
print(f"\nGeneralization verdict: {gen_verdict} (|Δ f1_macro| = {gap_macro:.4f})")

# Specifically flag OFFENSIVE degradation.
off_gap = dvt_rows[2]["Δ_abs"]
if off_gap < -0.04:
    print(f"⚠ F1_OFFENSIVE drops by {off_gap:+.4f} from dev to test — likely annotation ambiguity.")


## C. 3-way TEST comparison

Load Week-3 LR test predictions + metrics, run BiLSTM v2 on test once,
build the comparison table + figure.

In [ ]:
# Week-3 LR champion: predictions + metrics already produced.
LR_TEST_CSV  = RESULTS_DIR / "test_predictions.csv"
LR_METRICS_JSON = RESULTS_DIR / "week3_final_metrics.json"

lr_test_pred_df = pd.read_csv(LR_TEST_CSV)
print(f"LR test predictions: {len(lr_test_pred_df):,} rows  "
      f"(schema: {list(lr_test_pred_df.columns)})")

with open(LR_METRICS_JSON, "r", encoding="utf-8") as f:
    wk3 = json.load(f)
lr_test_metrics = wk3.get("test_metrics", {})
lr_champion_info = wk3.get("champion", {})
print(f"LR champion: {lr_champion_info.get('algorithm','?')} + {lr_champion_info.get('features','?')}")
print(f"  test f1_macro = {lr_test_metrics.get('f1_macro', 0):.4f}, "
      f"acc = {lr_test_metrics.get('accuracy', 0):.4f}")

# Sanity: LR test predictions and our test_df align?
assert len(lr_test_pred_df) == len(test_df), (
    f"Row count mismatch — LR has {len(lr_test_pred_df)}, test has {len(test_df)}; "
    "the LR file must come from the same test split as data/processed/test_cleaned.csv."
)


In [ ]:
from src.dataset_dl import Vocab, collate_fn_bilstm
from src.models_dl import BiLSTMClassifier

V2_CKPT = DL_DIR / "bilstm_v2_best.pt"
assert V2_CKPT.exists(), f"Missing {V2_CKPT} — run 04b2 first."

ck = torch.load(V2_CKPT, weights_only=False, map_location="cpu")
print(f"BiLSTM v2 checkpoint: experiment={ck.get('experiment','?')}, "
      f"best_epoch={ck.get('best_epoch','?')}, "
      f"dev_f1_macro={ck.get('metrics',{}).get('f1_macro',0):.4f}")

# Reconstruct model from checkpoint config. Strip non-model kwargs and the
# `pretrained_embeddings` tensor (we'll restore via load_state_dict instead).
cfg = ck["config"]
MODEL_KWARGS_KEYS = {
    "vocab_size", "embedding_dim", "hidden_dim", "num_layers", "num_classes",
    "dropout", "freeze_embeddings", "padding_idx", "head_dropout",
}
model_kwargs = {k: cfg[k] for k in MODEL_KWARGS_KEYS if k in cfg}
model_kwargs["pretrained_embeddings"] = None    # state_dict has the trained embeddings

bilstm_model = BiLSTMClassifier(**model_kwargs).to(device)
bilstm_model.load_state_dict(ck["model_state_dict"])
bilstm_model.eval()

BILSTM_PARAMS = sum(p.numel() for p in bilstm_model.parameters() if p.requires_grad)
print(f"  trainable params: {BILSTM_PARAMS:,}")

# Test dataset in BiLSTM mode (no filter — fair eval).
vocab = Vocab.load(DL_DIR / "vocab.pkl")
test_ds_lstm = ViHSDDataset(
    texts     = test_df["cleaned"].fillna("").tolist(),
    labels    = test_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = cfg.get("max_len", 64),
    mode      = "bilstm",
)
test_loader_lstm = DataLoader(
    test_ds_lstm, batch_size=128, shuffle=False,
    collate_fn=collate_fn_bilstm, num_workers=0, pin_memory=True,
)

bilstm_preds, bilstm_labels, bilstm_probs = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for batch in tqdm(test_loader_lstm, desc="bilstm test"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        lengths   = batch["lengths"]
        labels    = batch["labels"]
        logits = bilstm_model(input_ids, lengths)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        preds  = logits.argmax(dim=-1).cpu().numpy()
        bilstm_preds.extend(preds.tolist())
        bilstm_labels.extend(labels.numpy().tolist())
        bilstm_probs.extend(probs.tolist())
bilstm_test_time = time.perf_counter() - t0

y_test_pred_lstm  = np.asarray(bilstm_preds)
y_test_proba_lstm = np.asarray(bilstm_probs)
# Sanity: labels in same order as the canonical y_test_true (we don't shuffle the test loader).
assert np.array_equal(np.asarray(bilstm_labels), y_test_true), "BiLSTM label order ≠ PhoBERT label order"

print(f"\nBiLSTM v2 TEST inference: {bilstm_test_time:.2f}s on {len(y_test_pred_lstm):,} samples "
      f"({bilstm_test_time/len(y_test_pred_lstm)*1000:.2f} ms/sample)")

bilstm_test_metrics = evaluate_model(
    y_true=y_test_true, y_pred=y_test_pred_lstm,
    model_name="BiLSTM v2 (Exp3) — TEST",
    train_time=0.0, inference_time=float(bilstm_test_time),
)
print(f"\nBiLSTM v2 TEST metrics:")
print(f"  accuracy : {bilstm_test_metrics['accuracy']:.4f}")
print(f"  f1_macro : {bilstm_test_metrics['f1_macro']:.4f}")
print(f"  f1 clean / off / hate : "
      f"{bilstm_test_metrics['f1_clean']:.4f} / "
      f"{bilstm_test_metrics['f1_offensive']:.4f} / "
      f"{bilstm_test_metrics['f1_hate']:.4f}")

# Save BiLSTM v2 test predictions (same schema as PhoBERT).
bilstm_test_pred_df = pd.DataFrame({
    "text_original":    test_df[TEXT].values,
    "text_cleaned":     test_df["cleaned"].fillna("").values,
    "true_label":       y_test_true,
    "true_label_name":  [LABEL_MAP[int(l)] for l in y_test_true],
    "pred_label":       y_test_pred_lstm,
    "pred_label_name":  [LABEL_MAP[int(l)] for l in y_test_pred_lstm],
    "correct":          (y_test_true == y_test_pred_lstm),
    "confidence":       y_test_proba_lstm.max(axis=1).round(4),
    "prob_clean":       y_test_proba_lstm[:, 0].round(4),
    "prob_offensive":   y_test_proba_lstm[:, 1].round(4),
    "prob_hate":        y_test_proba_lstm[:, 2].round(4),
})
bilstm_test_csv = RESULTS_DIR / "bilstm_v2_test_predictions.csv"
bilstm_test_pred_df.to_csv(bilstm_test_csv, index=False)
print(f"\n✓ saved → {bilstm_test_csv}")

# Free GPU memory before the comparison plot.
del bilstm_model, phobert_model
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Training times: pull from each model's saved log (fall back to 0).
def _safe_load(p, default=None):
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

bilstm_log    = _safe_load(RESULTS_DIR / "bilstm_tuning_log.json", {})
phobert_log   = _safe_load(RESULTS_DIR / "phobert_training_log.json", {})

# Winner experiment from tuning log → its elapsed.
bilstm_train_time = 0.0
for r in bilstm_log.get("experiments", []):
    if r.get("name") == bilstm_log.get("winner"):
        bilstm_train_time = float(r.get("elapsed_s", 0.0))
        break

phobert_train_time = float(phobert_log.get("total_train_time", 0.0))
lr_train_time      = float(lr_test_metrics.get("train_time", 0.0))

# Build comparison rows (TEST-set metrics).
def _row(name, m, train_s, infer_s, params):
    return {
        "model":    name,
        "F1_macro": float(m["f1_macro"]),
        "F1_CLEAN": float(m["f1_clean"]),
        "F1_OFF":   float(m["f1_offensive"]),
        "F1_HATE":  float(m["f1_hate"]),
        "Acc":      float(m["accuracy"]),
        "train_s":  float(train_s),
        "infer_s":  float(infer_s),
        "params":   int(params),
    }

LR_PARAMS = int(wk3.get("champion", {}).get("n_features_total", 25_000))
test_rows = [
    _row("LR + char champion (Wk 3)", lr_test_metrics, lr_train_time, lr_test_metrics.get("inference_time", 0.0), LR_PARAMS),
    _row("BiLSTM v2 (Exp3)",          bilstm_test_metrics, bilstm_train_time, bilstm_test_time, BILSTM_PARAMS),
    _row("PhoBERT-base-v2",           phobert_test_metrics, phobert_train_time, phobert_test_time, PHOBERT_PARAMS),
]
test_compare_df = pd.DataFrame(test_rows)

print("3-way comparison (TEST):\n")
print(test_compare_df.to_string(
    index=False,
    formatters={
        "F1_macro":  "{:.4f}".format,
        "F1_CLEAN":  "{:.4f}".format,
        "F1_OFF":    "{:.4f}".format,
        "F1_HATE":   "{:.4f}".format,
        "Acc":       "{:.4f}".format,
        "train_s":   "{:.1f}".format,
        "infer_s":   "{:.2f}".format,
        "params":    "{:,d}".format,
    }))

# Champion = row with max F1_macro.
champion_row = max(test_rows, key=lambda r: r["F1_macro"])
print(f"\n★ TEST CHAMPION: {champion_row['model']}  (F1_macro = {champion_row['F1_macro']:.4f})")

# Deltas vs LR on TEST.
lr_row_t = test_rows[0]
phb_row_t = test_rows[2]
bl_row_t  = test_rows[1]
print(f"\nPhoBERT Δ vs LR on TEST:")
print(f"  f1_macro: {phb_row_t['F1_macro']-lr_row_t['F1_macro']:+.4f}")
print(f"  f1_OFF  : {phb_row_t['F1_OFF']-lr_row_t['F1_OFF']:+.4f}")
print(f"  f1_HATE : {phb_row_t['F1_HATE']-lr_row_t['F1_HATE']:+.4f}")
print(f"\nPhoBERT Δ vs BiLSTM v2 on TEST:")
print(f"  f1_macro: {phb_row_t['F1_macro']-bl_row_t['F1_macro']:+.4f}")
print(f"  f1_OFF  : {phb_row_t['F1_OFF']-bl_row_t['F1_OFF']:+.4f}")
print(f"  f1_HATE : {phb_row_t['F1_HATE']-bl_row_t['F1_HATE']:+.4f}")


In [ ]:
# Big 3-panel figure: F1 macro bar + per-class bar + cost-benefit + 3 cms
fig = plt.figure(figsize=(15, 11), constrained_layout=True)
gs  = fig.add_gridspec(3, 3)

names_short = ["LR", "BiLSTM v2", "PhoBERT"]
palette     = ["#9D7BBA", "#4C78A8", "#54A24B"]
classes     = ["CLEAN", "OFFENSIVE", "HATE"]
f1m         = [r["F1_macro"] for r in test_rows]

# (top-left) F1 macro bar
ax = fig.add_subplot(gs[0, 0])
bars = ax.bar(range(3), f1m, color=palette)
ax.set_xticks(range(3)); ax.set_xticklabels(names_short)
ax.set_ylim(0.5, max(0.8, max(f1m) + 0.05))
ax.set_ylabel("Test F1 macro"); ax.set_title("F1 macro (TEST)")
for bar, val in zip(bars, f1m):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.4f}",
            ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")

# (top-mid+right) per-class grouped bar
ax = fig.add_subplot(gs[0, 1:])
x = np.arange(len(classes)); w = 0.27
for i, r in enumerate(test_rows):
    vals = [r["F1_CLEAN"], r["F1_OFF"], r["F1_HATE"]]
    ax.bar(x + (i - 1) * w, vals, w, label=names_short[i], color=palette[i])
ax.set_xticks(x); ax.set_xticklabels(classes)
ax.set_ylabel("F1"); ax.set_title("Per-class F1 (TEST)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3, axis="y")

# (mid-left) cost-benefit: log(params) vs F1
ax = fig.add_subplot(gs[1, 0])
xs = [max(r["params"], 1) for r in test_rows]
ax.scatter(xs, f1m, c=palette, s=160, edgecolor="white", linewidth=1.5, zorder=3)
for x_, y_, n in zip(xs, f1m, names_short):
    ax.annotate(n, (x_, y_), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Trainable params (log)"); ax.set_ylabel("Test F1 macro")
ax.set_title("Params vs F1 (TEST)")
ax.grid(alpha=0.3, which="both")

# (mid-mid) train time vs F1
ax = fig.add_subplot(gs[1, 1])
xs2 = [max(r["train_s"], 0.1) for r in test_rows]
ax.scatter(xs2, f1m, c=palette, s=160, edgecolor="white", linewidth=1.5, zorder=3)
for x_, y_, n in zip(xs2, f1m, names_short):
    ax.annotate(n, (x_, y_), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Train time (s, log)"); ax.set_ylabel("Test F1 macro")
ax.set_title("Train-time vs F1 (TEST)")
ax.grid(alpha=0.3, which="both")

# (mid-right) summary text
ax = fig.add_subplot(gs[1, 2]); ax.axis("off")
ax.text(0.0, 1.0, "TEST headline deltas:", fontsize=11, weight="bold", va="top")
text = (
    f"PhoBERT vs LR:\n"
    f"  ΔF1_macro = {phb_row_t['F1_macro']-lr_row_t['F1_macro']:+.4f}\n"
    f"  ΔF1_OFF   = {phb_row_t['F1_OFF']-lr_row_t['F1_OFF']:+.4f}\n"
    f"  ΔF1_HATE  = {phb_row_t['F1_HATE']-lr_row_t['F1_HATE']:+.4f}\n\n"
    f"PhoBERT vs BiLSTM v2:\n"
    f"  ΔF1_macro = {phb_row_t['F1_macro']-bl_row_t['F1_macro']:+.4f}\n"
    f"  ΔF1_OFF   = {phb_row_t['F1_OFF']-bl_row_t['F1_OFF']:+.4f}\n"
    f"  ΔF1_HATE  = {phb_row_t['F1_HATE']-bl_row_t['F1_HATE']:+.4f}\n\n"
    f"PhoBERT generalisation:\n"
    f"  Δ dev→test F1m = {phobert_test_metrics['f1_macro']-phobert_dev_metrics['f1_macro']:+.4f}\n"
    f"  ({gen_verdict})"
)
ax.text(0.0, 0.92, text, fontsize=9.5, family="monospace", va="top")

# (bottom row) 3 confusion matrices side by side
# LR uses predicted_label, BiLSTM/PhoBERT use stored arrays from this notebook.
y_lr_pred = lr_test_pred_df["predicted_label"].to_numpy()
for j, (name, color, y_pred) in enumerate(zip(
        names_short, palette,
        [y_lr_pred, y_test_pred_lstm, y_test_pred_phb])):
    ax = fig.add_subplot(gs[2, j])
    cm = confusion_matrix(y_test_true, y_pred, labels=[0, 1, 2])
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(classes, fontsize=8)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(classes, fontsize=8)
    ax.set_title(f"{name}\nF1m={test_rows[j]['F1_macro']:.4f}", fontsize=10)
    for ii in range(3):
        for jj in range(3):
            ax.text(jj, ii, f"{cm_norm[ii,jj]:.1%}\n({cm[ii,jj]})",
                    ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[ii,jj] > 0.5 else "black")
    ax.set_xlabel("predicted")
    if j == 0:
        ax.set_ylabel("true")

fig.suptitle("Week 4 — TEST comparison: LR vs BiLSTM v2 vs PhoBERT-base-v2",
             fontsize=13, y=1.01)
fig_path = FIG_DIR / "22_three_way_test_comparison.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


## D. Final summary + JSON for downstream

Auto-generates `report/week4_final_summary.md` and a JSON dump
`results/week4_test_metrics.json` that Weeks 5 / 6 can read.

In [ ]:
# Disagreement set sizes for Week-5 hints.
phb_correct = phobert_test_pred_df["correct"].to_numpy()
lr_correct  = (lr_test_pred_df["predicted_label"].to_numpy() == y_test_true)
lstm_correct = bilstm_test_pred_df["correct"].to_numpy()

# LR right & PhoBERT wrong
lr_right_phb_wrong = int(((~phb_correct) & lr_correct).sum())
# PhoBERT right & LR wrong
phb_right_lr_wrong = int((phb_correct & (~lr_correct)).sum())
# All 3 wrong
all_wrong = int(((~phb_correct) & (~lr_correct) & (~lstm_correct)).sum())

# OFFENSIVE → CLEAN false negatives for each model.
def _off_to_clean(y_true, y_pred):
    mask = (y_true == 1) & (y_pred == 0)
    return int(mask.sum()), int((y_true == 1).sum())
lr_o2c   = _off_to_clean(y_test_true, y_lr_pred)
lstm_o2c = _off_to_clean(y_test_true, y_test_pred_lstm)
phb_o2c  = _off_to_clean(y_test_true, y_test_pred_phb)

# Build markdown.
gap_macro_signed = phobert_test_metrics["f1_macro"] - phobert_dev_metrics["f1_macro"]

lines = []
lines.append("# Week 4 — Final Summary (TEST evaluation)")
lines.append("")
lines.append("_Auto-generated by `notebooks/04d_test_evaluation.ipynb`. Test set "
             "touched **once** here; no tuning after this point._")
lines.append("")
lines.append("## 1. Pipeline progression")
lines.append("")
lines.append("| Stage | Approach | F1_macro DEV | F1_macro TEST |")
lines.append("|---|---|---|---|")
lines.append(f"| 1 | LR baseline (Week 3) | {wk3.get('dev_metrics',{}).get('f1_macro',0):.4f} | — |")
lines.append(f"| 2 | LR + char-ngram tuned (Week 3 champion) | "
             f"{wk3.get('dev_metrics',{}).get('f1_macro',0):.4f} | "
             f"{lr_test_metrics.get('f1_macro',0):.4f} |")
# BiLSTM baseline (04b) f1
bilstm_baseline_f1 = bilstm_log.get("baseline", {}).get("best", {}).get("f1_macro", 0.5999)
lines.append(f"| 3 | BiLSTM baseline (overfit) | {bilstm_baseline_f1:.4f} | — |")
lines.append(f"| 4 | BiLSTM v2 (Exp3 strong reg) | "
             f"{ck.get('metrics',{}).get('f1_macro',0):.4f} | "
             f"{bilstm_test_metrics['f1_macro']:.4f} |")
lines.append(f"| 5 | **PhoBERT-base-v2 fine-tuned** | "
             f"**{phobert_dev_metrics['f1_macro']:.4f}** | "
             f"**{phobert_test_metrics['f1_macro']:.4f}** |")
lines.append("")

lines.append("## 2. Champion (TEST)")
lines.append("")
lines.append(f"- **Model**: {champion_row['model']}")
lines.append(f"- **Test F1 macro**: {champion_row['F1_macro']:.4f}")
lines.append(f"- **Test F1 OFFENSIVE**: {champion_row['F1_OFF']:.4f}")
lines.append(f"- **Test F1 HATE**: {champion_row['F1_HATE']:.4f}")
lines.append(f"- **Test accuracy**: {champion_row['Acc']:.4f}")
lines.append("")

lines.append("## 3. 3-way TEST comparison")
lines.append("")
lines.append("| Model | F1 macro | F1 CLEAN | F1 OFF | F1 HATE | Accuracy | Train | Inference | Params |")
lines.append("|---|---|---|---|---|---|---|---|---|")
for r in test_rows:
    lines.append(
        f"| {r['model']} "
        f"| {r['F1_macro']:.4f} "
        f"| {r['F1_CLEAN']:.4f} "
        f"| {r['F1_OFF']:.4f} "
        f"| {r['F1_HATE']:.4f} "
        f"| {r['Acc']:.4f} "
        f"| {r['train_s']:.0f}s "
        f"| {r['infer_s']:.2f}s "
        f"| {r['params']:,} |"
    )
lines.append("")

lines.append("## 4. PhoBERT dev vs test generalisation")
lines.append("")
lines.append("| Metric | Dev | Test | Δ abs | Δ rel |")
lines.append("|---|---|---|---|---|")
for row in dvt_rows:
    lines.append(
        f"| {row['Metric']} "
        f"| {row['Dev']:.4f} "
        f"| {row['Test']:.4f} "
        f"| {row['Δ_abs']:+.4f} "
        f"| {row['Δ_rel_pct']:+.2f}% |"
    )
lines.append("")
lines.append(f"**Verdict**: {gen_verdict} (|Δ f1_macro| = {abs(gap_macro_signed):.4f}).")
lines.append("")

lines.append("## 5. Key findings")
lines.append("")
lines.append("1. **Three-tier hierarchy**: static features (LR + char) < static embeddings (BiLSTM v2) "
             "< contextual embeddings (PhoBERT). Each tier brings monotonic improvement on TEST f1_macro.")
lines.append(f"2. BiLSTM needed strong regularisation (LSTM dropout 0.5, head dropout 0.6, "
             f"weight_decay 1e-3) to beat LR by only +{bl_row_t['F1_macro']-lr_row_t['F1_macro']:+.4f} — "
             f"static word embeddings are at their ceiling on this dataset.")
lines.append(f"3. PhoBERT improves OFFENSIVE F1 by {phb_row_t['F1_OFF']-lr_row_t['F1_OFF']:+.4f} and "
             f"HATE F1 by {phb_row_t['F1_HATE']-lr_row_t['F1_HATE']:+.4f} — the two minority classes "
             f"where contextual sub-word embeddings + transformer attention pay off most.")
lines.append(f"4. **Persistent failure mode**: OFFENSIVE → CLEAN false negatives are common to all 3 models:")
lines.append(f"   - LR: {lr_o2c[0]} / {lr_o2c[1]} OFFENSIVE samples ({lr_o2c[0]/max(lr_o2c[1],1)*100:.1f}%)")
lines.append(f"   - BiLSTM v2: {lstm_o2c[0]} / {lstm_o2c[1]} ({lstm_o2c[0]/max(lstm_o2c[1],1)*100:.1f}%)")
lines.append(f"   - PhoBERT: {phb_o2c[0]} / {phb_o2c[1]} ({phb_o2c[0]/max(phb_o2c[1],1)*100:.1f}%)")
lines.append(f"   The fact that this rate is similar across vastly different model classes points to "
             f"**annotation ambiguity** in the OFFENSIVE category rather than a modelling gap.")
lines.append("")

lines.append("## 6. Failure modes — hypotheses for Week 5 analysis")
lines.append("")
lines.append("- **H1 (annotation ambiguity)**: OFFENSIVE samples with subtle sarcasm or in-group banter "
             "may be borderline-labeled; rough estimate of agreement: per (4), ~"
             f"{phb_o2c[0]/max(phb_o2c[1],1)*100:.0f}% of OFFENSIVE crosses the CLEAN boundary even for PhoBERT.")
lines.append("- **H2 (short-comment difficulty)**: comments with length ≤ 2 (filtered out of TRAINING but "
             "kept in DEV/TEST) carry teencode/slurs that the model never trained against.")
lines.append("- **H3 (code-switched English-Vietnamese slurs)**: PhoBERT-v2's vocab is Vietnamese-centric; "
             "English profanity may decompose into uninformative sub-words.")
lines.append("- **H4 (confidence-calibration mismatch)**: high-confidence wrong predictions are usually "
             "the most informative class — `phobert_test_predictions.csv` has the probabilities.")
lines.append("")

lines.append("## 7. Artefacts for Week 5")
lines.append("")
lines.append(f"- `results/test_predictions.csv` — LR test predictions (Week 3 schema: `predicted_label`, `proba_*`)")
lines.append(f"- `results/bilstm_v2_test_predictions.csv` — BiLSTM v2 test predictions (Week 4 schema)")
lines.append(f"- `results/phobert_test_predictions.csv` — PhoBERT test predictions (Week 4 schema) — **champion file**")
lines.append(f"- `models/dl/phobert_best/` — final champion model + tokenizer")
lines.append(f"- `results/week4_test_metrics.json` — all numbers above, machine-readable")
lines.append("")

lines.append("## 8. Bridge to Week 5: Error analysis tasks")
lines.append("")
lines.append("1. **Disagreement sets** (loaded across the 3 prediction CSVs by row order):")
lines.append(f"   - LR right ∧ PhoBERT wrong: {lr_right_phb_wrong} cases — what does LR catch that PhoBERT misses?")
lines.append(f"   - PhoBERT right ∧ LR wrong: {phb_right_lr_wrong} cases — what's the contextual signal?")
lines.append(f"   - All 3 wrong: {all_wrong} cases — likely annotation issues or out-of-distribution.")
lines.append("2. **Confidence calibration**: bin PhoBERT `confidence` into deciles, plot accuracy per bin. "
             "If high-confidence wrongs are concentrated in OFFENSIVE → CLEAN, that's H1 evidence.")
lines.append("3. **Length stratification**: split test by length quantile, look at f1_macro per bucket. "
             "Validates H2 about short comments.")
lines.append("4. **Linguistic categories**: tag a sample of failures by sarcasm / code-switching / teencode / "
             "all-caps; cross with H3.")

summary_path = REPORT_DIR / "week4_final_summary.md"
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ saved → {summary_path}\n")
print(summary_path.read_text(encoding="utf-8"))


In [ ]:
# Machine-readable dump for Week 5 / Week 6.
test_metrics_payload = {
    "test_set_size":     int(len(y_test_true)),
    "class_distribution": {
        LABEL_MAP[i]: int((y_test_true == i).sum()) for i in (0, 1, 2)
    },
    "models": {
        "lr_champion": {
            "name":     test_rows[0]["model"],
            "test":     {k: float(v) for k, v in lr_test_metrics.items()
                          if isinstance(v, (int, float))},
            "params":   int(LR_PARAMS),
            "predictions_csv": str(LR_TEST_CSV.relative_to(ROOT)),
        },
        "bilstm_v2": {
            "name":     test_rows[1]["model"],
            "test":     {
                "accuracy":         float(bilstm_test_metrics["accuracy"]),
                "f1_macro":         float(bilstm_test_metrics["f1_macro"]),
                "f1_clean":         float(bilstm_test_metrics["f1_clean"]),
                "f1_offensive":     float(bilstm_test_metrics["f1_offensive"]),
                "f1_hate":          float(bilstm_test_metrics["f1_hate"]),
                "precision_macro":  float(bilstm_test_metrics["precision_macro"]),
                "recall_macro":     float(bilstm_test_metrics["recall_macro"]),
            },
            "params":             int(BILSTM_PARAMS),
            "test_inference_s":   float(bilstm_test_time),
            "train_s":            float(bilstm_train_time),
            "predictions_csv":    str(bilstm_test_csv.relative_to(ROOT)),
            "checkpoint":         str(V2_CKPT.relative_to(ROOT)),
        },
        "phobert": {
            "name":     test_rows[2]["model"],
            "test":     {
                "accuracy":         float(phobert_test_metrics["accuracy"]),
                "f1_macro":         float(phobert_test_metrics["f1_macro"]),
                "f1_clean":         float(phobert_test_metrics["f1_clean"]),
                "f1_offensive":     float(phobert_test_metrics["f1_offensive"]),
                "f1_hate":          float(phobert_test_metrics["f1_hate"]),
                "precision_macro":  float(phobert_test_metrics["precision_macro"]),
                "recall_macro":     float(phobert_test_metrics["recall_macro"]),
            },
            "dev":      {
                "accuracy":         float(phobert_dev_metrics["accuracy"]),
                "f1_macro":         float(phobert_dev_metrics["f1_macro"]),
                "f1_clean":         float(phobert_dev_metrics["f1_clean"]),
                "f1_offensive":     float(phobert_dev_metrics["f1_offensive"]),
                "f1_hate":          float(phobert_dev_metrics["f1_hate"]),
            },
            "dev_vs_test":        dvt_rows,
            "generalization_verdict": gen_verdict,
            "params":             int(PHOBERT_PARAMS),
            "test_inference_s":   float(phobert_test_time),
            "train_s":            float(phobert_train_time),
            "predictions_csv":    str(phobert_test_csv.relative_to(ROOT)),
            "model_dir":          str(PHOBERT_BEST_DIR.relative_to(ROOT)),
        },
    },
    "champion": {
        "model":     champion_row["model"],
        "f1_macro":  float(champion_row["F1_macro"]),
    },
    "disagreement_set_sizes": {
        "lr_right_phobert_wrong":  lr_right_phb_wrong,
        "phobert_right_lr_wrong":  phb_right_lr_wrong,
        "all_three_wrong":         all_wrong,
    },
    "offensive_to_clean_fn": {
        "lr":         {"count": lr_o2c[0],   "total": lr_o2c[1]},
        "bilstm_v2":  {"count": lstm_o2c[0], "total": lstm_o2c[1]},
        "phobert":    {"count": phb_o2c[0],  "total": phb_o2c[1]},
    },
}

out_json = RESULTS_DIR / "week4_test_metrics.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(test_metrics_payload, f, indent=2, ensure_ascii=False)
print(f"✓ saved → {out_json}")

# Final reminder.
print(f"\nWeek 4 evaluation complete. Champion on TEST: {champion_row['model']}  "
      f"F1_macro = {champion_row['F1_macro']:.4f}.")
print("Next: Week 5 error analysis using results/phobert_test_predictions.csv.")
